# Kannolo Dense Index Benchmark (Euclidean)

## Import Required Libraries

In [ ]:
import importlib.metadata as md

print("kannolo", md.version("kannolo"))

In [ ]:
import kannolo
import numpy as np
import time
from pathlib import Path

## Load and Prepare Dense Dataset

In [ ]:
# Load dense dataset (adjust path as needed): a 2-D float array (n, dim)
data_path = ""
data = np.load(data_path).astype(np.float32)
print(f"Dataset shape: {data.shape}")
n_data, dim = data.shape
print(f"Number of data points: {n_data}, Dimension: {dim}")

print("")

# Load dense queries: a 2-D float array (n_queries, dim)
queries_path = ""
queries = np.load(queries_path).astype(np.float32)
print(f"Queries shape: {queries.shape}")
n_queries = queries.shape[0]

## Create Flat Index and Compute Ground Truth

In [ ]:
# Exhaustive (flat) index under squared Euclidean distance: exact neighbors
flat_index = kannolo.build(data, metric="l2", index=kannolo.Flat())
print("Flat index built")

# Compute ground truth (k=10): ids of shape (n_queries, k)
k = 10
_, groundtruth = flat_index.search(queries, k)
print(f"Ground truth computed for {n_queries} queries, k={k}")

## Build HNSW Index

In [ ]:
# HNSW with m=32 neighbors and ef_construction=200, vectors stored in f16
start = time.time()
hnsw = kannolo.build(data, metric="l2", encoder=kannolo.Plain("f16"), index=kannolo.HNSW(m=32, ef_construction=200))
end = time.time()
print(f"HNSW index built in {end - start} seconds")
print(hnsw)

## Search with Multiple Parameter Combinations

In [ ]:
# Parameter configurations to test
configs = [
    {"ef_search": 20, "early_exit_threshold": None},
    {"ef_search": 100, "early_exit_threshold": None},
    {"ef_search": 15, "early_exit_threshold": 0.1}
]

results = []

for config in configs:
    ef_search = config["ef_search"]
    early_exit = config["early_exit_threshold"]

    # Search (single-threaded, to time per-query latency)
    start = time.time()
    distances, results_ids = hnsw.search(queries, k, ef_search=ef_search, early_exit_threshold=early_exit, num_threads=1)
    elapsed = time.time() - start

    # Compute accuracy: results and ground truth are (n_queries, k) arrays
    matches = sum(len(set(gt) & set(res)) for gt, res in zip(groundtruth, results_ids))
    accuracy = (matches / (n_queries * k)) * 100
    avg_time = (elapsed / n_queries) * 10**6  # us per query

    results.append({
        "ef_search": ef_search,
        "early_exit_threshold": early_exit,
        "avg_time_us": avg_time,
        "accuracy": accuracy
    })

    print(f"ef_search={ef_search}, early_exit={early_exit}: "
          f"avg_time={avg_time:.3f}us, accuracy={accuracy:.1f}%")

print("\n" + "="*60)
for r in results:
    print(f"Config: ef_search={r['ef_search']}, early_exit={r['early_exit_threshold']}")
    print(f"  Avg Query Time: {r['avg_time_us']:.3f} us")
    print(f"  Accuracy: {r['accuracy']:.1f}%")
    print()